In [2]:
!pip install langchain langchain-community pypdf sentence_transformers faiss-cpu langchain-anthropic

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from langchain_community.document_loaders import PyPDFLoader

document_url = "PrintingBasics.pdf"
loader = PyPDFLoader(document_url)
pages = loader.load()

In [4]:
print(pages[0].page_content[0:250])

 
1 
Table of Contents 
Navigating the online manual  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .2 
Printing Basics. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 
How to choose paper. . . . . . . . . . . . . .


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(pages)
print(chunks[0])

page_content='1 
Table of Contents 
Navigating the online manual  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .2 
Printing Basics. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 
How to choose paper. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .3' metadata={'source': 'PrintingBasics.pdf', 'page': 0}


In [6]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

chunk_texts = list(map(lambda d: d.page_content, chunks))
embeddings = bge_embeddings.embed_documents(chunk_texts)
print(embeddings[0])

C:\Users\neelam.bhanuprakash\AppData\Roaming\Python\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[-0.06221699342131615, -0.029451778158545494, 0.02954699657857418, -0.026096627116203308, 0.03240951895713806, 0.021043043583631516, -0.0232740119099617, 0.042582955211400986, -0.03521648421883583, 0.00648993207141757, -0.003648698329925537, -0.0012097111903131008, 0.0013583002146333456, -0.0023486497811973095, 0.030257660895586014, 0.010443469509482384, 0.018781455233693123, -0.02140044793486595, 0.010413878597319126, 0.04169468581676483, 0.049559395760297775, -0.005757184699177742, -0.024015335366129875, -0.02260681428015232, 0.01933811418712139, 0.047079265117645264, -0.018375879153609276, -0.018081871792674065, -0.010903425514698029, -0.1855696588754654, 0.01183241605758667, -0.0006110554095357656, 0.023717211559414864, 0.007982643321156502, 0.004873123485594988, -0.0483112670481205, 0.005023912992328405, 0.00629494758322835, 0.002519741654396057, -0.021273711696267128, -0.013158478774130344, 0.0018222552025690675, -0.003086946438997984, -0.03148864582180977, 0.032940011471509933, 

In [7]:
from langchain_community.vectorstores import FAISS

text_embedding_pairs = zip(chunk_texts, embeddings)
db = FAISS.from_embeddings(text_embedding_pairs, bge_embeddings)

In [8]:
query = "About pictures?"

contexts = db.similarity_search(query, k=5)

print(contexts[0])

page_content='15 
About pictures 
Pictures (also called  
graphics ) include photographs, illustrations, 
charts, and decorative elements. 
How to get a picture on your computer'
